In [30]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [31]:
pip install -U google-genai


Note: you may need to restart the kernel to use updated packages.


In [32]:
google_api_key=os.getenv("GOOGLE_API_KEY")

In [33]:
from google import genai

client = genai.Client(api_key=google_api_key)

response = client.models.generate_content(
    model="gemini-3-flash-preview", contents="Explain how AI works in a few words"
)
print(response.text)

AI learns **patterns from data** to make **predictions or decisions.**


In [34]:
from google.genai import types
import pathlib

# Retrieve and encode the PDF byte
filepath = pathlib.Path("C:/Users/shanm/Downloads/AI_ML_DL_NLP_Overview.pdf")

prompt = "Summarize this document"
response = client.models.generate_content(
  model="gemini-3-flash-preview",
  contents=[
      types.Part.from_bytes(
        data=filepath.read_bytes(),
        mime_type='application/pdf',
      ),
      prompt])
print(response.text)

This document provides a foundational overview of Artificial Intelligence (AI) and its primary subfields. It is structured into four main sections:

*   **Artificial Intelligence (AI):** Defined as the simulation of human intelligence in machines to perform tasks like reasoning and problem-solving. It is categorized into **Narrow AI** (task-specific) and **General AI** (human-level intelligence).
*   **Machine Learning (ML):** A subset of AI that enables systems to learn from data patterns without explicit programming. It includes techniques such as supervised, unsupervised, and reinforcement learning, commonly used for fraud detection and recommendation engines.
*   **Deep Learning (DL):** A specialized subset of ML that utilizes artificial neural networks with multiple hidden layers. It excels at learning from raw data for complex tasks like image recognition and autonomous driving using architectures like CNNs and Transformers.
*   **Natural Language Processing (NLP):** A field of A

In [36]:
pip install PyPDF2


Note: you may need to restart the kernel to use updated packages.


## step 1 : Extract text from PDF

In [37]:
from PyPDF2 import PdfReader
import pathlib

filepath = pathlib.Path("C:/Users/shanm/Downloads/AI_ML_DL_NLP_Overview.pdf")

reader = PdfReader(filepath)
pdf_text = ""

for page in reader.pages:
    pdf_text += page.extract_text() + "\n"

print(len(pdf_text))  # sanity check


2091


In [38]:
filepath = pathlib.Path("C:/Users/shanm/Downloads/AI_ML_DL_NLP_Overview.pdf")
result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=pdf_text,
    config=types.EmbedContentConfig(output_dimensionality=768)
)

[embedding_obj] = result.embeddings
embedding_length = len(embedding_obj.values)

print(f"Length of embedding: {embedding_length}")

Length of embedding: 768


## STEP2 : TEXT CHUNKING

In [41]:
EMBED_DIM = 768
CHUNK_SIZE = 500    
CHUNK_OVERLAP = 100
def chunk_text(text, size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = chunk_text(pdf_text, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"Created {len(chunks)} chunks")

Created 6 chunks


## step 3 : create embeddings

In [48]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context


In [49]:
import numpy as np

In [50]:
EMBED_MODEL = "gemini-embedding-001"
embeddings = []
for i, chunk in enumerate(chunks):
    response = client.models.embed_content(
        model=EMBED_MODEL,
        contents=chunk,
        config=types.EmbedContentConfig(
            output_dimensionality=EMBED_DIM
        )
    )
    embeddings.append(response.embeddings[0].values)

embeddings = np.array(embeddings).astype("float32")
print(f"Generated embeddings shape: {embeddings.shape}")

Generated embeddings shape: (6, 768)


## step 4 : STORE IN FAISS INDEX

In [55]:
pip install faiss-cpu -f https://download.pytorch.org/whl/cpu

Looking in links: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.9 MB 2.0 MB/s eta 0:00:10
   - -------------------------------------- 0.8/18.9 MB 2.1 MB/s eta 0:00:09
   -- ------------------------------------- 1.0/18.9 MB 1.7 MB/s eta 0:00:11
   -- ------------------------------------- 1.3/18.9 MB 1.7 MB/s eta 0:00:11
   --- ------------------------------------ 1.6/18.9 MB 1.4 MB/s eta 0:00:13
   ------ --------------------------------- 2.9/18.9 MB 2.2 MB/s eta 0:00:08
   ---------- ----------------------------- 5.0/18.9 MB 3.3 MB/s eta 0:00:05
   ------------ --------------------------- 5.8/18.9 MB 3.6 MB/s eta 0:00:04
   ------------ --------------------------- 6.0/18.9 MB 3.4 MB/s eta 0:00:04
   ------------ --------------------------- 6.0/18.9 MB 3.4 MB/s eta 0:00:04
   ------------- -------------------------- 6.3/18.9 MB 2.9 MB/s eta 0:00:05
   ------------- -----------

In [56]:
import faiss


In [ ]:
index = faiss.IndexFlatL2(EMBED_DIM)
index.add(embeddings)
print(f"✅ FAISS index size: {index.ntotal}")

## step 5 : SAVE INDEX + METADATA

In [59]:
faiss.write_index(index, "pdf_vector.index")

with open("chunks.txt", "w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(chunk.replace("\n", " ") + "\n")

print("Vector index + chunks saved successfully!")

Vector index + chunks saved successfully!


##  VERIFY that vectors are really there

In [61]:
index = faiss.read_index("pdf_vector.index")
print("Number of vectors:", index.ntotal)
print("Vector dimension:", index.d)

Number of vectors: 6
Vector dimension: 768


In [62]:
vec = index.reconstruct(0)
print(vec)
print(len(vec))


[-7.70442141e-03  1.93591006e-02  2.15743799e-02 -3.97997163e-02
 -3.42983678e-02 -1.29146501e-03 -2.58979457e-03 -6.89978246e-04
  2.65382626e-03 -1.98476557e-02 -1.75849684e-02  2.42804792e-02
 -6.59826398e-03  2.24030633e-02  1.20855667e-01  1.94655992e-02
 -1.29204933e-02  1.73912048e-02  2.39413492e-02 -2.48303656e-02
 -2.71780114e-03 -7.67454971e-03 -6.31069532e-03 -1.19167380e-02
 -4.13808003e-02 -1.32865971e-03 -4.39121854e-03 -5.30094141e-03
  2.90689953e-02  6.79767225e-03  5.46091190e-03  1.68640111e-02
  8.03410076e-03  2.79231332e-02  4.25131666e-03  1.22819957e-03
  9.39935911e-03 -2.89427023e-02 -1.06961350e-03  2.89246794e-02
 -3.56904343e-02 -4.64533875e-03 -2.48714816e-02  5.84241794e-03
 -1.19134393e-02  2.09438205e-02  1.30578503e-02 -2.62367520e-02
 -1.91756058e-02  1.48159219e-02 -1.95928756e-03  7.37251015e-03
  3.97007167e-03 -1.61809802e-01 -1.34617640e-02 -3.37814307e-03
 -1.08255763e-02  7.67626474e-03  2.90513765e-02 -9.11581703e-03
 -3.22304741e-02 -1.14898

In [63]:
import faiss
import numpy as np
from google import genai
from google.genai import types

# -----------------------------
# CONFIG
# -----------------------------
INDEX_PATH = "pdf_vector.index"
CHUNKS_PATH = "chunks.txt"
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
TOP_K = 3

# -----------------------------
# INIT GEMINI CLIENT
# -----------------------------
client = genai.Client()

# -----------------------------
# LOAD FAISS INDEX
# -----------------------------
index = faiss.read_index(INDEX_PATH)
print("Loaded FAISS index")
print("Vectors:", index.ntotal, "| Dim:", index.d)

# -----------------------------
# LOAD CHUNKS
# -----------------------------
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = f.readlines()

print("  text chunks:", len(chunks))


Loaded FAISS index
Vectors: 6 | Dim: 768
  text chunks: 6


## EMBED USER QUERY

In [64]:
def embed_query(query: str):
    response = client.models.embed_content(
        model=EMBED_MODEL,
        contents=query,
        config=types.EmbedContentConfig(
            output_dimensionality=EMBED_DIM
        )
    )
    return np.array(response.embeddings[0].values, dtype="float32")


## Search FAISS index

In [65]:
def search_index(query: str, top_k=TOP_K):
    query_vec = embed_query(query)
    query_vec = np.expand_dims(query_vec, axis=0)

    distances, indices = index.search(query_vec, top_k)

    results = []
    for idx in indices[0]:
        results.append(chunks[idx].strip())

    return results


## Test retrieval only (IMPORTANT)

In [66]:
query = "Explain the difference between machine learning and deep learning"

results = search_index(query)

print("Retrieved Context:\n")
for i, text in enumerate(results, 1):
    print(f"[{i}] {text}\n")


Retrieved Context:

[1] sed on those patterns. Common types of machine learning include supervised learning, unsupervised learning, and reinforcement learning. ML is widely used in spam detection, recommendation engines, fraud detection, and predictive analytics.  Deep Learning (DL)  Deep Learning is a specialized subset of machine learning that uses artificial neural networks with multiple hidden layers. These deep neural networks are capable of automatically learning hierarchical feature representations from raw data

[2] l assistants. AI can be broadly categorized into Narrow AI, which performs specific tasks, and General AI, which aspires to perform any intellectual task that a human can do.  Machine Learning (ML)  Machine Learning is a subset of AI that focuses on enabling machines to learn from data without being explicitly programmed. ML algorithms identify patterns in data and make predictions or decisions based on those patterns. Common types of machine learning include superv

In [70]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [74]:
def ask_pdf(query: str):
    context_chunks = search_index(query)

    context = "\n\n".join(context_chunks)

    prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer clearly and concisely.
"""

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )

    return response.text



In [75]:
question = "What is the relationship between AI, ML, and Deep Learning?"

answer = ask_pdf(question)

print("Answer:\n")
print(answer)

Answer:

Based on the context provided, the relationship between AI, ML, and Deep Learning is a nested hierarchy:

*   **Artificial Intelligence (AI):** The broad field focused on enabling machines to perform intellectual tasks.
*   **Machine Learning (ML):** A subset of AI that enables machines to learn from data and identify patterns without being explicitly programmed.
*   **Deep Learning (DL):** A specialized subset of Machine Learning that uses artificial neural networks with multiple hidden layers to automatically learn feature representations from raw data.
